# Asia-Pacífico: Ensambladores Sofisticados en la Era de la IA
## Una crítica empírica al *EAP Economic Update* (Banco Mundial, Abril 2026)

**Eduard Romero | edalytics | Mayo 2026**

---

> *¿Puede un país liderar las exportaciones tecnológicas globales y al mismo tiempo quedarse fuera de la revolución que exporta?*
> Los datos dicen que sí — y este análisis lo demuestra.

---


## 1. Pregunta de investigación e hipótesis

El informe *East Asia & Pacific Economic Update* (Banco Mundial, abril 2026) revela 
un hecho llamativo: países como Malaysia y Vietnam exportan productos relacionados 
con inteligencia artificial equivalentes a un tercio de su PIB, mientras que la 
adopción interna de IA en sus empresas apenas alcanza el 13–17%, frente al 37% 
registrado en Estados Unidos.

Esta paradoja — ser locomotora de exportación tecnológica sin subirse al propio tren — 
plantea una pregunta de investigación con implicaciones directas de política económica:

> *¿En qué medida las exportaciones tecnológicas impulsan el crecimiento en 
> Asia-Pacífico, y por qué la política industrial no ha logrado traducir ese 
> dinamismo exportador en adopción interna de IA?*

Para responderla de forma estructurada, el análisis se organiza en torno a tres 
hipótesis complementarias:

| | Hipótesis | Bloque de análisis |
|---|---|---|
| **H1** | Las exportaciones de productos tecnológicos tienen un efecto positivo y significativo sobre el crecimiento del PIB en los países de EAP | Crecimiento |
| **H2** | Existe una brecha estadísticamente significativa entre la intensidad exportadora tecnológica y la adopción interna de IA entre países | Paradoja |
| **H3** | Los incentivos fiscales tienen mayor efecto sobre la productividad que los subsidios directos en economías emergentes de EAP | Política industrial |

**Nota metodológica:** H2 y H3 se abordan de forma descriptiva apoyándose en el 
informe del Banco Mundial y fuentes externas (Stanford HAI, OpenDOSM), dado que 
no existe una base de datos pública estructurada que permita su contraste 
econométrico formal. H1 se contrasta econométricamente con datos de panel del WDI.


## 2. Datos

### 2.1 Fuentes y variables

El análisis econométrico de **H1** utiliza datos del *World Development Indicators* 
(WDI) del Banco Mundial, accesibles directamente desde Python via `wbgapi`. 
Esto garantiza la reproducibilidad completa del análisis.

**Muestra:** 6 países de Asia-Pacífico — China, Malaysia, Vietnam, Filipinas, 
Indonesia y Camboya.  
**Período:** 2010–2023.

| Variable | Rol | Código WDI |
|---|---|---|
| Crecimiento del PIB per cápita (%) | Dependiente | `NY.GDP.PCAP.KD.ZG` |
| Exportaciones de alta tecnología (% exportaciones manufacturadas) | Independiente principal | `TX.VAL.TECH.MF.ZS` |
| Formación bruta de capital fijo (% PIB) | Control | `NE.GDI.FTOT.ZS` |
| Inflación IPC (%) | Control | `FP.CPI.TOTL.ZG` |
| Apertura comercial (X+M / PIB) | Control | `NE.TRD.GNFS.ZS` |
| Variación tipo de cambio (%) | Control | `PA.NUS.FCRF` |


### 2.2 Configuración del entorno

In [1]:
import wbgapi as wb
import pandas as pd
import numpy as np
import requests
import plotly.graph_objects as go
from linearmodels.panel import PanelOLS, RandomEffects, FirstDifferenceOLS, compare
from scipy import stats

# ── Paleta edalytics ──────────────────────────────────────────────────────────
COLORS = ['#2c3e50', '#e74c3c', '#27ae60', '#3498db', '#e67e22', '#9b59b6']

# ── Estilo base reutilizable ──────────────────────────────────────────────────
BASE_LAYOUT = dict(
    font=dict(family='Georgia, serif', size=12, color='#2c3e50'),
    paper_bgcolor='#fafaf8',
    plot_bgcolor='#fafaf8',
    xaxis=dict(
        gridcolor='#e8e8e4',
        gridwidth=0.5,
        linecolor='#2c3e50',
        showgrid=False,
        tickfont=dict(color='#2c3e50')
    ),
    yaxis=dict(
        gridcolor='#e8e8e4',
        gridwidth=0.8,
        linecolor='#2c3e50',
        tickfont=dict(color='#2c3e50'),
        zeroline=True,
        zerolinecolor='#2c3e50',
        zerolinewidth=1
    ),
    legend=dict(
        bgcolor='#fafaf8',
        bordercolor='#e8e8e4',
        borderwidth=1,
        font=dict(color='#2c3e50')
    ),
    margin=dict(t=110, b=60)
)

def add_title(fig, title, subtitle):
    fig.add_annotation(
        text=f'<b>{title}</b>',
        xref='paper', yref='paper',
        x=0, y=1.15,
        showarrow=False,
        font=dict(size=15, color='#2c3e50', family='Georgia, serif'),
        align='left'
    )
    fig.add_annotation(
        text=subtitle,
        xref='paper', yref='paper',
        x=0, y=1.08,
        showarrow=False,
        font=dict(size=11, color='#888', family='Georgia, serif'),
        align='left'
    )
    fig.add_shape(
        type='line',
        xref='paper', yref='paper',
        x0=0, x1=0.45, y0=1.03, y1=1.03,
        line=dict(color='#2c3e50', width=2)
    )

def add_source(fig, source='World Bank WDI | edalytics.com'):
    fig.add_annotation(
        text=f'Fuente: {source}',
        xref='paper', yref='paper',
        x=0, y=-0.12,
        showarrow=False,
        font=dict(size=10, color='#888'),
        align='left'
    )

print("Entorno configurado ✓")


Entorno configurado ✓


### 2.3 Descarga de datos

In [2]:
indicators = {
    'NY.GDP.PCAP.KD.ZG': 'gdp_growth',
    'TX.VAL.TECH.MF.ZS': 'tech_exports',
    'NE.GDI.FTOT.ZS'   : 'investment',
    'FP.CPI.TOTL.ZG'   : 'inflation',
    'NE.TRD.GNFS.ZS'   : 'trade_openness'
}

countries = ['CHN', 'MYS', 'VNM', 'PHL', 'IDN', 'KHM']

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy=countries,
    time=range(2010, 2024),
    labels=True
).reset_index()

print(f"Dimensiones: {df.shape}")


Dimensiones: (30, 18)


### 2.4 Transformación a formato long

In [3]:
df_long = (
    df.melt(
        id_vars=['economy', 'series', 'Country', 'Series'],
        var_name='year',
        value_name='value'
    )
    .assign(year=lambda x: x['year'].str.replace('YR', '').astype(int))
    .pivot_table(
        index=['economy', 'Country', 'year'],
        columns='Series',
        values='value'
    )
    .reset_index()
)

df_long.columns.name = None

df_long = df_long.rename(columns={
    'GDP per capita growth (annual %)'                    : 'gdp_growth',
    'Gross fixed capital formation (% of GDP)'            : 'investment',
    'High-technology exports (% of manufactured exports)' : 'tech_exports',
    'Inflation, consumer prices (annual %)'               : 'inflation',
    'Trade (% of GDP)'                                    : 'trade_openness'
})

df_long.head()


,economy,Country,year,gdp_growth,investment,tech_exports,inflation,trade_openness
0,CHN,China,2010,10.063424,43.520182,32.150117,3.175325,49.854072
1,CHN,China,2011,8.864807,43.515391,30.500480,5.553899,49.945827
2,CHN,China,2012,7.127013,43.873708,30.861809,2.619524,47.480213
3,CHN,China,2013,7.063225,44.075543,31.585581,2.621050,45.916041
4,CHN,China,2014,6.786670,43.380347,29.703727,1.921642,44.068456


### 2.5 Tipo de cambio nominal (variación anual)

In [4]:
er = wb.data.DataFrame(
    'PA.NUS.FCRF',
    economy=['CHN', 'MYS', 'VNM', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
).reset_index()

er_long = (
    er.melt(
        id_vars=['economy', 'Country'],
        var_name='year',
        value_name='exchange_rate'
    )
    .assign(year=lambda x: x['year'].str.replace('YR', '').astype(int))
    .sort_values(['Country', 'year'])
)

er_long['der'] = er_long.groupby('Country')['exchange_rate'].pct_change() * 100

df_long = df_long.merge(
    er_long[['economy', 'year', 'der']],
    on=['economy', 'year'],
    how='left'
)

print("Tipo de cambio incorporado ✓")


Tipo de cambio incorporado ✓


### 2.6 Nota sobre valores faltantes

Filipinas (`PHL`) no reporta datos de exportaciones de alta tecnología 
(`tech_exports`) para el período 2010–2016. A partir de 2017 la serie 
está completa.

Se opta por mantener el **panel desbalanceado** — decisión válida 
econométricamente y consistente con la práctica habitual en datos de panel 
con series incompletas. El estimador de efectos fijos de `linearmodels` 
maneja nativamente observaciones faltantes sin necesidad de imputación.


## 3. Estadísticos descriptivos

### 3.1 Resumen general

In [5]:
df_long.describe().round(2)

,year,gdp_growth,investment,tech_exports,inflation,trade_openness,der
count,84.00,84.00,84.00,77.00,84.00,84.00,78.00
mean,2016.50,4.38,29.75,29.14,3.38,93.04,1.83
std,4.06,2.98,6.98,20.82,2.46,47.06,4.45
min,2010.00,-10.55,18.21,0.11,-1.14,32.97,-6.55
25%,2013.00,3.69,24.80,8.45,2.09,47.29,-0.51
50%,2016.50,4.77,29.84,30.82,2.94,91.11,0.98
75%,2020.00,6.08,32.39,47.57,3.89,132.73,3.76
max,2023.00,10.06,44.08,67.05,18.68,186.68,19.33


La muestra cubre 84 observaciones país-año (77 para `tech_exports` por 
los valores faltantes de Filipinas documentados en la sección anterior).

Varios patrones merecen atención:

- **Crecimiento:** media de 4.38% anual, consistente con el dinamismo característico 
  de las economías emergentes de la región. El mínimo de -10.55% corresponde al 
  impacto del COVID-19 en 2020.
- **Exportaciones tecnológicas:** desviación típica de 20.82 puntos — la heterogeneidad 
  entre países es elevada, lo cual favorece la identificación econométrica.
- **Apertura comercial:** rango de 33% a 187% del PIB, reflejo de la diferencia 
  estructural entre China y Vietnam.
- **Inflación:** máximo de 18.68% — valor atípico identificado en el análisis por país.


### 3.2 Estadísticos por país

In [6]:
df_long.groupby('Country')[['gdp_growth', 'investment', 'tech_exports',
                             'inflation', 'trade_openness']].mean().round(2)


,gdp_growth,investment,tech_exports,inflation,trade_openness
Country,,,,,
Cambodia,4.60,28.38,2.49,3.15,123.88
China,6.42,42.29,30.33,2.25,40.38
Indonesia,3.64,31.60,9.12,4.20,43.04
Malaysia,2.92,23.12,51.35,2.04,136.89
Philippines,3.77,22.77,63.68,3.45,63.55
Viet Nam,4.93,30.31,35.17,5.16,150.52


La tabla revela heterogeneidad estructural significativa entre países:

- **Filipinas y Malaysia** lideran en exportaciones tecnológicas (63% y 51%) pero 
  registran los crecimientos medios más bajos de la muestra (3.77% y 2.92%) — una 
  primera señal de la paradoja que articula el análisis.
- **China** destaca por una tasa de inversión media del 42% del PIB, reflejo de un 
  modelo de crecimiento históricamente basado en acumulación de capital físico.
- **Vietnam** presenta una apertura comercial del 150% del PIB, confirmando su rol 
  como hub exportador altamente integrado en cadenas de valor regionales.
- **Camboya** muestra el nivel más bajo de exportaciones tecnológicas (2.49%) pero 
  mantiene un crecimiento sólido, sustentado principalmente en manufactura textil 
  y turismo.


### 3.3 Evolución temporal

**Figura 1 — Crecimiento del PIB per cápita**

In [7]:
countries_list = df_long['Country'].unique()

fig1 = go.Figure()

for i, country in enumerate(countries_list):
    d = df_long[df_long['Country'] == country]
    fig1.add_trace(go.Scatter(
        x=d['year'], y=d['gdp_growth'],
        name=country, mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig1.update_layout(**BASE_LAYOUT, title=None)
add_title(fig1, 'Crecimiento del PIB per cápita',
          'Asia-Pacífico, 2010–2023 | % anual')
fig1.update_yaxes(ticksuffix='%')
add_source(fig1)
fig1.show()


El gráfico revela tres patrones estructurales en el crecimiento de la región 
durante el período 2010–2023.

**Convergencia a la baja pre-COVID.** China lidera el crecimiento en 2010 con un 
10.06% pero desacelera de forma sostenida hasta situarse en torno al 6% en 
2017–2019, convergiendo hacia las tasas del resto de economías de la muestra.

**El shock de 2020.** La pandemia impacta de forma asimétrica: Filipinas registra 
la caída más severa (-10.55%), seguida de Malaysia (-6.71%) e Indonesia (-2.89%). 
Vietnam y Camboya muestran una resiliencia notable.

**Recuperación heterogénea post-2021.** Vietnam y Malaysia lideran el rebote en 
2022 con crecimientos de 7.73%, mientras China se frena abruptamente hasta el 3.15%.


**Figura 2 — Exportaciones de alta tecnología**

In [8]:
fig2 = go.Figure()

for i, country in enumerate(countries_list):
    d = df_long[df_long['Country'] == country]
    fig2.add_trace(go.Scatter(
        x=d['year'], y=d['tech_exports'],
        name=country, mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig2.update_layout(**BASE_LAYOUT, title=None)
add_title(fig2, 'Exportaciones de alta tecnología',
          'Asia-Pacífico, 2010–2023 | % exportaciones manufacturadas')
fig2.update_yaxes(ticksuffix='%')
add_source(fig2)
fig2.show()


El gráfico de exportaciones de alta tecnología expone una de las heterogeneidades 
estructurales más marcadas de la región y constituye el punto de partida empírico 
de la paradoja que articula este análisis.

**Dos clubes tecnológicos.** Malaysia y Filipinas operan en una franja alta y estable 
— entre el 47% y el 67% — mientras Indonesia y Camboya se mantienen por debajo 
del 13% durante prácticamente todo el período.

**Vietnam: la trayectoria más destacada.** Vietnam pasa del 13% en 2010 al 44% en 
2023 — un incremento de 31 puntos porcentuales en trece años.

**La paradoja.** Filipinas lidera en intensidad exportadora tecnológica desde 2017 
con valores superiores al 60%, pero registra uno de los crecimientos del PIB per 
cápita más moderados de la muestra.


**Figura 3 — Apertura comercial**

In [9]:
fig3 = go.Figure()

for i, country in enumerate(countries_list):
    d = df_long[df_long['Country'] == country]
    fig3.add_trace(go.Scatter(
        x=d['year'], y=d['trade_openness'],
        name=country, mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig3.update_layout(**BASE_LAYOUT, title=None)
add_title(fig3, 'Apertura comercial',
          'Asia-Pacífico, 2010–2023 | (X+M)/PIB en %')
fig3.update_yaxes(ticksuffix='%')
add_source(fig3)
fig3.show()


La evolución de la apertura comercial revela dos modelos estructurales claramente 
diferenciados. Vietnam, Malaysia y Cambodia superan el 100% del PIB en comercio 
exterior durante todo el período — en el caso de Vietnam alcanza el 187% en 2021. 
China e Indonesia muestran una tendencia secular a la baja, con China pasando del 
49.9% en 2011 al 34% en 2020.


**Figura 4 — Matriz de correlaciones**

In [10]:
corr_matrix = df_long[['gdp_growth', 'tech_exports', 'investment',
                         'inflation', 'trade_openness', 'der']].corr().round(2)

fig_heat = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale=[[0.0, '#e74c3c'], [0.5, '#fafaf8'], [1.0, '#2c3e50']],
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text}',
    textfont=dict(size=11, family='Georgia, serif'),
    showscale=True
))

fig_heat.update_layout(**BASE_LAYOUT, title=None)
add_title(fig_heat, 'Matriz de correlaciones', 'Variables del panel | Pearson')
add_source(fig_heat, 'World Bank WDI | Estimación propia')
fig_heat.show()


La correlación más elevada del panel es entre `investment` y `trade_openness` 
(r = -0.42), reflejo de la diferencia estructural entre China (alta inversión, baja 
apertura) y Vietnam o Cambodia (baja inversión, alta apertura). `tech_exports` 
correlaciona negativamente con el crecimiento (r = -0.13) — primera señal de la 
paradoja que el modelo econométrico cuantificará.


## 4. Especificación econométrica

### 4.1 Modelo

El análisis empírico de **H1** se basa en un modelo de datos de panel con la 
siguiente especificación:

$$gdp\_growth_{it} = \alpha_i + \beta_1 tech\_exports_{it} + \beta_2 investment_{it} + \beta_3 trade\_openness_{it} + \varepsilon_{it}$$

Donde:

- $i$ indexa los países y $t$ los años
- $\alpha_i$ recoge la heterogeneidad no observada específica de cada país
- $\varepsilon_{it}$ es el término de error idiosincrático

La elección entre efectos fijos y efectos aleatorios se determina empíricamente 
mediante el **test de Hausman**. Se estiman tres especificaciones progresivas para 
evaluar la robustez de los resultados.


### 4.2 Preparación del panel

In [11]:
# Panel en niveles
df_panel = df_long.set_index(['Country', 'year'])

# Panel en primeras diferencias
diff_vars = ['gdp_growth', 'tech_exports', 'investment', 'trade_openness']

df_diff = (
    df_long
    .sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [diff_vars]
    .groupby(level='Country')
    .diff()
    .dropna()
)

print("Panel en niveles:", df_panel.shape)
print("Panel en diferencias:", df_diff.shape)


Panel en niveles: (84, 7)
Panel en diferencias: (71, 4)


### 4.3 Estimación y diagnósticos

In [12]:
# ── Modelo 1: Efectos fijos en niveles ───────────────────────────────────────
fe_model = PanelOLS(
    dependent=df_panel['gdp_growth'],
    exog=df_panel[['tech_exports', 'investment', 'inflation', 'trade_openness']],
    entity_effects=True
)
fe_result = fe_model.fit(cov_type='clustered', cluster_entity=True)

# ── Modelo 2: Efectos aleatorios (para test Hausman) ─────────────────────────
re_model = RandomEffects(
    dependent=df_panel['gdp_growth'],
    exog=df_panel[['tech_exports', 'investment', 'inflation', 'trade_openness']]
)
re_result = re_model.fit()

# ── Test de Hausman (manual) ──────────────────────────────────────────────────
b_diff = fe_result.params - re_result.params
v_diff = fe_result.cov - re_result.cov
H = float(b_diff.T @ np.linalg.inv(v_diff) @ b_diff)
p_hausman = 1 - stats.chi2.cdf(H, len(b_diff))

print(f"Test de Hausman — Estadístico: {H:.4f} | P-valor: {p_hausman:.4f}")
print("→ Se rechaza efectos aleatorios al 5%" if p_hausman < 0.05 else "→ No se rechaza efectos aleatorios")

# ── Autocorrelación serial ────────────────────────────────────────────────────
resid_df = fe_result.resids.reset_index()
resid_df.columns = ['Country', 'year', 'resid']
resid_df['resid_lag'] = resid_df.groupby('Country')['resid'].shift(1)
corr_serial = resid_df[['resid', 'resid_lag']].dropna().corr().iloc[0, 1]
print(f"Autocorrelación serial de residuos: {corr_serial:.4f}")


Test de Hausman — Estadístico: 10.1905 | P-valor: 0.0373
→ Se rechaza efectos aleatorios al 5%
Autocorrelación serial de residuos: 0.0575


/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning:


Inputs contain missing values. Dropping rows with missing observations.

/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:2751: MissingValueWarning:


Inputs contain missing values. Dropping rows with missing observations.



In [13]:
# ── Modelo 3: Primeras diferencias completo ──────────────────────────────────
fd_model_full = FirstDifferenceOLS(
    dependent=df_diff['gdp_growth'],
    exog=df_diff[['tech_exports', 'investment', 'trade_openness']]
)
fd_result = fd_model_full.fit(cov_type='clustered', cluster_entity=True)

# ── Modelo 4: Efectos umbral (threshold en 30%) ───────────────────────────────
df_long['high_tech'] = (df_long['tech_exports'] >= 30).astype(int)
df_long['tech_high'] = df_long['tech_exports'] * df_long['high_tech']

df_diff_b = (
    df_long.sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [['gdp_growth', 'tech_exports', 'investment', 'trade_openness', 'tech_high']]
    .groupby(level='Country').diff().dropna()
)

fd_model_threshold = FirstDifferenceOLS(
    dependent=df_diff_b['gdp_growth'],
    exog=df_diff_b[['tech_exports', 'investment', 'trade_openness', 'tech_high']]
)
fd_result_threshold = fd_model_threshold.fit(cov_type='clustered', cluster_entity=True)

# ── Modelo 5: Interacción política industrial ─────────────────────────────────
df_long['policy'] = df_long['Country'].apply(
    lambda x: 1 if x in ['China', 'Malaysia'] else 0
)
df_long['tech_policy'] = df_long['tech_exports'] * df_long['policy']

df_diff_d = (
    df_long.sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [['gdp_growth', 'tech_exports', 'investment', 'trade_openness', 'tech_policy']]
    .groupby(level='Country').diff().dropna()
)

fd_model_policy = FirstDifferenceOLS(
    dependent=df_diff_d['gdp_growth'],
    exog=df_diff_d[['tech_exports', 'investment', 'trade_openness', 'tech_policy']]
)
fd_result_policy = fd_model_policy.fit(cov_type='clustered', cluster_entity=True)

print("Todos los modelos estimados ✓")


Todos los modelos estimados ✓


### 4.4 Comparación de modelos

In [14]:
print(compare({
    'FE Niveles'          : fe_result,
    'FD Parsimonioso'     : fd_result,
    'FD Umbral'           : fd_result_threshold,
    'FD Política Ind.'    : fd_result_policy
}))


                                              Model Comparison                                             
                            FE Niveles        FD Parsimonioso              FD Umbral       FD Política Ind.
-----------------------------------------------------------------------------------------------------------
Dep. Variable               gdp_growth             gdp_growth             gdp_growth             gdp_growth
Estimator                     PanelOLS     FirstDifferenceOLS     FirstDifferenceOLS     FirstDifferenceOLS
No. Observations                    77                     65                     65                     65
Cov. Est.                    Clustered              Clustered              Clustered              Clustered
R-squared                       0.1297                 0.3959                 0.4301                 0.4184
R-Squared (Within)              0.1297                 0.1593                 0.1424                 0.1236
R-Squared (Between)         

### 4.5 Resultados e interpretación

#### Diagnósticos

- **Test de Hausman:** H = 10.19 (p = 0.037) — se rechaza efectos aleatorios al 5%. 
  Los efectos individuales están correlacionados con los regresores.
- **Autocorrelación serial:** ρ = 0.057 — prácticamente nula. Los errores 
  clusterizados por entidad son suficientes.
- **Especificación:** la inconsistencia entre variable dependiente en tasas y 
  regresores en niveles motiva la estimación en primeras diferencias como 
  especificación preferida.

#### Tabla de resultados

| Variable | FE Niveles | FD Parsimonioso | FD Umbral | FD Política Ind. |
|---|---|---|---|---|
| `tech_exports` | -0.131 | -0.868*** | -1.029*** | -1.124** |
| `investment` | -0.043 | 0.730 | 0.808 | 0.736 |
| `trade_openness` | 0.093** | 0.320** | 0.294** | 0.294** |
| `tech_high` | — | — | 0.112** | — |
| `tech_policy` | — | — | — | 0.773 |
| R² | 0.130 | 0.412 | 0.430 | 0.418 |

*Errores clusterizados por país. \*p<0.10, \*\*p<0.05, \*\*\*p<0.01*

#### Interpretación económica

**`trade_openness`** es el único regresor consistentemente significativo a través 
de todas las especificaciones. Un incremento de 10 puntos porcentuales en la ratio 
comercio/PIB se asocia con un aumento de 3.2pp en el crecimiento del PIB per cápita.

**`tech_exports`** presenta un coeficiente negativo y significativo al 1% en el 
modelo parsimonioso (-0.868). Este resultado contraintuitivo es coherente con la 
paradoja central del análisis: los países que más tecnología exportan — Malaysia y 
Filipinas — operan principalmente como ensambladores en cadenas de valor globales, 
con escaso valor añadido doméstico.

**El efecto umbral** (FD Umbral) es significativo al 5% (0.112): superar el 30% 
de intensidad exportadora tecnológica atenúa el efecto negativo pero no lo revierte.

#### Limitaciones

El R² within de 0.41 indica que el modelo explica una fracción moderada de la 
variación intra-país. Con N=6 los estimadores son imprecisos y los resultados deben 
interpretarse con cautela como evidencia descriptiva más que causal.


### 4.6 Visualización de resultados

**Figura 5 — Coeficientes del modelo parsimonioso**

In [15]:
coef_df = pd.DataFrame({
    'coef' : fd_result.params,
    'lower': fd_result.conf_int()['lower'],
    'upper': fd_result.conf_int()['upper']
}).reset_index()
coef_df.columns = ['variable', 'coef', 'lower', 'upper']

labels = {
    'tech_exports'  : 'Exportaciones tecnológicas',
    'investment'    : 'Inversión',
    'trade_openness': 'Apertura comercial'
}
coef_df['variable'] = coef_df['variable'].map(labels)
coef_df['color'] = coef_df.apply(
    lambda r: '#e74c3c' if r['lower'] > 0 or r['upper'] < 0 else '#95a5a6', axis=1
)

fig_coef = go.Figure()

for _, row in coef_df.iterrows():
    fig_coef.add_trace(go.Scatter(
        x=[row['lower'], row['upper']], y=[row['variable'], row['variable']],
        mode='lines', line=dict(color=row['color'], width=2), showlegend=False
    ))
    fig_coef.add_trace(go.Scatter(
        x=[row['coef']], y=[row['variable']],
        mode='markers', marker=dict(color=row['color'], size=10), showlegend=False
    ))

fig_coef.add_vline(x=0, line=dict(color='#2c3e50', width=1, dash='dash'))
fig_coef.update_layout(**BASE_LAYOUT, title=None)
fig_coef.update_xaxes(title='Coeficiente')
add_title(fig_coef, 'Coeficientes del modelo parsimonioso',
          'Intervalos de confianza al 95% | Errores clusterizados por país')
add_source(fig_coef, 'World Bank WDI | Estimación propia')
fig_coef.show()


**Figura 6 — Scatter exportaciones tecnológicas vs crecimiento (en diferencias)**

In [16]:
df_scatter = df_diff.reset_index().merge(
    df_long[['Country', 'economy', 'year']],
    on=['Country', 'year'], how='left'
).dropna(subset=['tech_exports', 'gdp_growth'])

fig_scatter = go.Figure()

for i, country in enumerate(df_scatter['Country'].unique()):
    d = df_scatter[df_scatter['Country'] == country]
    fig_scatter.add_trace(go.Scatter(
        x=d['tech_exports'], y=d['gdp_growth'],
        mode='markers', name=country,
        marker=dict(color=COLORS[i], size=8, opacity=0.8)
    ))

x_vals = df_scatter['tech_exports']
y_vals = df_scatter['gdp_growth']
z = np.polyfit(x_vals, y_vals, 1)
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)

fig_scatter.add_trace(go.Scatter(
    x=x_line, y=np.poly1d(z)(x_line),
    mode='lines', name='Tendencia',
    line=dict(color='#2c3e50', width=1.5, dash='dash')
))

fig_scatter.update_layout(**BASE_LAYOUT, title=None)
fig_scatter.update_xaxes(title='Δ Exportaciones tecnológicas (pp)')
fig_scatter.update_yaxes(title='Crecimiento PIB per cápita (%)', ticksuffix='%')
add_title(fig_scatter, 'Variación en exportaciones tecnológicas y crecimiento',
          'Países EAP, 2011–2023 | Cada punto es un país-año')
add_source(fig_scatter, 'World Bank WDI | Estimación propia')
fig_scatter.show()


**Figura 7 — Diagnóstico: valores ajustados vs residuos**

In [17]:
fitted = fd_result.fitted_values.values.flatten()
resids = fd_result.resids.values.flatten()

fig_diag = go.Figure()
fig_diag.add_trace(go.Scatter(
    x=fitted, y=resids,
    mode='markers',
    marker=dict(color='#2c3e50', size=7, opacity=0.7),
    showlegend=False
))
fig_diag.add_hline(y=0, line=dict(color='#e74c3c', width=1.5, dash='dash'))

fig_diag.update_layout(**BASE_LAYOUT, title=None)
fig_diag.update_xaxes(title='Valores ajustados')
fig_diag.update_yaxes(title='Residuos')
add_title(fig_diag, 'Diagnóstico: valores ajustados vs residuos',
          'Modelo parsimonioso en primeras diferencias')
add_source(fig_diag, 'Estimación propia')
fig_diag.show()


## 5. Crítica al informe del Banco Mundial

### 5.1 ¿Exportar tecnología equivale a innovar?

El informe utiliza las exportaciones tecnológicas como proxy de capacidad 
tecnológica. Si esa proxy fuera válida, debería correlacionar fuertemente 
con innovación doméstica — medida por patentes registradas. Los datos muestran 
que no es así.


### 5.2 Descarga de datos de patentes (WDI)

In [18]:
patents = wb.data.DataFrame(
    'IP.PAT.RESD',
    economy=['CHN', 'MYS', 'VNM', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
).reset_index()

patents_long = (
    patents
    .melt(id_vars=['economy', 'Country'], var_name='year', value_name='patents')
    .assign(year=lambda x: x['year'].str.replace('YR', '').astype(int))
    .query('year <= 2021')
    .query('Country != "Cambodia"')
    .dropna(subset=['patents'])
)

patents_long['log_patents'] = np.log(patents_long['patents'])

df_long = df_long.merge(
    patents_long[['economy', 'year', 'log_patents']],
    on=['economy', 'year'], how='left'
)

print("Patentes incorporadas ✓")


Patentes incorporadas ✓


**Figura 8 — ¿Exportar tecnología equivale a innovar?**

In [19]:
df_critic = (
    df_long
    .query('Country != "China"')
    .dropna(subset=['tech_exports', 'log_patents'])
    .groupby('Country')[['tech_exports', 'log_patents']]
    .mean()
    .reset_index()
)

fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']], y=[row['log_patents']],
        mode='markers+text', name=row['Country'],
        text=[row['Country']], textposition='top center',
        marker=dict(color=COLORS[i], size=12), showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic, '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Excluye China')
add_source(fig_critic, 'World Bank WDI | Estimación propia')
fig_critic.show()


El scatter confirma la hipótesis crítica: **Malaysia exporta 5 veces más 
tecnología que Indonesia pero innova exactamente igual** — ambos países tienen 
log_patents en torno a 7.0. Filipinas lidera en exportaciones tecnológicas (63%) 
pero tiene el nivel de innovación doméstica más bajo de la muestra.

Esta desconexión entre exportación tecnológica e innovación doméstica es la 
evidencia más directa de que los países de EAP operan como ensambladores 
sofisticados, no como economías tecnológicamente capaces.


### 5.3 Productividad del sector tecnológico en Malaysia
*(Datos primarios — OpenDOSM, Department of Statistics Malaysia)*


In [20]:
import pandas as pd

# Productividad laboral trimestral
URL_PROD = 'https://storage.dosm.gov.my/labour/productivity_qtr.parquet'
URL_LOOKUP = 'https://storage.dosm.gov.my/labour/productivity_lookup.parquet'

df_prod = pd.read_parquet(URL_PROD)
df_prod['date'] = pd.to_datetime(df_prod['date'])

df_lookup = pd.read_parquet(URL_LOOKUP)

# Sectores relevantes
tech_sectors = ['p3.7', 'p0']

df_tech = (
    df_prod
    .query("series == 'abs'")
    .query("sector in @tech_sectors")
    .merge(df_lookup[['code', 'desc_en']], left_on='sector', right_on='code', how='left')
    [['date', 'sector', 'desc_en', 'output_hour']]
    .sort_values(['sector', 'date'])
)

print("OpenDOSM cargado ✓")
print(df_tech.groupby('desc_en')['date'].agg(['min', 'max']))


OpenDOSM cargado ✓
                                                   min        max
desc_en                                                          
Electrical, electronic and optical products 2015-01-01 2025-10-01
Overall                                     2015-01-01 2025-10-01


**Figura 9 — Productividad laboral: sector tecnológico vs economía general**

In [21]:
fig_prod = go.Figure()

for i, (sector, label) in enumerate([('p0', 'Economía general'),
                                      ('p3.7', 'Eléctrico y electrónico')]):
    d = df_tech[df_tech['sector'] == sector]
    fig_prod.add_trace(go.Scatter(
        x=d['date'], y=d['output_hour'],
        name=label, mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig_prod.update_layout(**BASE_LAYOUT, title=None)
fig_prod.update_yaxes(title='Valor añadido por hora trabajada (MYR)')
add_title(fig_prod, 'Productividad laboral en Malaysia: ¿lidera la tecnología?',
          'Valor añadido por hora trabajada | 2015–2025')
add_source(fig_prod, 'OpenDOSM — Department of Statistics Malaysia')
fig_prod.show()


**Figura 10 — Crecimiento YoY de productividad**

In [22]:
df_tech_yoy = (
    df_tech
    .sort_values(['sector', 'date'])
    .assign(yoy=lambda x: x.groupby('sector')['output_hour'].pct_change(4) * 100)
    .dropna(subset=['yoy'])
)

fig_yoy = go.Figure()

for i, (sector, label) in enumerate([('p0', 'Economía general'),
                                      ('p3.7', 'Eléctrico y electrónico')]):
    d = df_tech_yoy[df_tech_yoy['sector'] == sector]
    fig_yoy.add_trace(go.Scatter(
        x=d['date'], y=d['yoy'],
        name=label, mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig_yoy.add_hline(y=0, line=dict(color='#2c3e50', width=1, dash='dash'))
fig_yoy.update_layout(**BASE_LAYOUT, title=None)
fig_yoy.update_yaxes(title='Crecimiento YoY (%)', ticksuffix='%')
add_title(fig_yoy, 'Crecimiento de productividad laboral en Malaysia',
          'Variación anual | 2016–2025')
add_source(fig_yoy, 'OpenDOSM — Department of Statistics Malaysia')
fig_yoy.show()


**Figura 11 — Brecha de productividad**

In [23]:
df_gap = (
    df_tech
    .pivot_table(index='date', columns='sector', values='output_hour')
    .reset_index()
    .assign(gap=lambda x: x['p3.7'] - x['p0'])
)

fig_gap = go.Figure()
fig_gap.add_trace(go.Scatter(
    x=df_gap['date'], y=df_gap['gap'],
    mode='lines', fill='tozeroy',
    fillcolor='rgba(44, 62, 80, 0.15)',
    line=dict(color='#2c3e50', width=2),
    showlegend=False
))
fig_gap.add_hline(y=0, line=dict(color='#e74c3c', width=1.5, dash='dash'))

fig_gap.update_layout(**BASE_LAYOUT, title=None)
fig_gap.update_yaxes(title='Diferencia en MYR por hora trabajada')
add_title(fig_gap, 'Brecha de productividad: tecnología vs economía general',
          'Sector eléctrico-electrónico menos media nacional | Malaysia 2015–2025')
add_source(fig_gap, 'OpenDOSM — Department of Statistics Malaysia')
fig_gap.show()


Durante 2023, el sector eléctrico y electrónico de Malaysia registró cuatro 
trimestres consecutivos de caída de productividad — llegando al -8.22% en Q4 2023 — 
precisamente el año en que las exportaciones tecnológicas del país alcanzaban el 
58% del total manufacturado. La brecha de productividad, que creció hasta 43.2 
MYR/hora en Q1 2022, se ha contraído hasta 30–34 MYR/hora desde entonces.

**Malaysia exporta más tecnología cada año pero su sector tecnológico no es más 
productivo.** Esto es inconsistente con un proceso genuino de upgrading tecnológico 
y consistente con un modelo de ensamblaje sofisticado sin spillovers productivos internos.


### 5.4 La trampa estadística del Banco Mundial: ¿qué mide realmente la adopción de IA?

El informe cita cifras de adopción interna de IA del 13–17% en subsidiarias de 
multinacionales en China y Tailandia. El *Stanford HAI AI Index 2025*, basado en 
la encuesta global de McKinsey, ofrece una imagen radicalmente distinta.

La diferencia es la **unidad de medida**:

- **Banco Mundial:** adopción en subsidiarias de multinacionales — las empresas más 
  avanzadas tecnológicamente por definición
- **McKinsey/Stanford:** adopción en organizaciones en general

Si el 13–17% es el techo de adopción en las mejores empresas de la región, ¿qué 
está pasando en las empresas locales? El BM presenta el techo como si fuera la media.


**Figura 12 — BM vs McKinsey: dos métricas, dos realidades**

In [24]:
fig_contrast = go.Figure()

fig_contrast.add_trace(go.Bar(
    x=['Banco Mundial\n(subsidiarias multinacionales)',
       'McKinsey/Stanford HAI\n(organizaciones en general)'],
    y=[15, 77],
    marker_color=['#e74c3c', '#2c3e50'],
    text=['13–17%', '77%'],
    textposition='outside',
    textfont=dict(size=14, family='Georgia, serif', color='#2c3e50'),
    showlegend=False,
    width=0.4
))

fig_contrast.update_layout(**BASE_LAYOUT, title=None)
fig_contrast.update_yaxes(title='% adopción de IA', ticksuffix='%', range=[0, 100])
add_title(fig_contrast,
          '¿Quién tiene razón sobre la adopción de IA en Asia-Pacífico?',
          'Banco Mundial (2026) vs McKinsey/Stanford HAI (2025) | Asia-Pacífico')
add_source(fig_contrast,
           'Banco Mundial EAP Economic Update 2026 | Stanford HAI AI Index 2025')
fig_contrast.show()


**Figura 13 — Adopción organizacional de IA por región**

In [25]:
df_adoption = pd.DataFrame({
    'Region': ['Asia-Pacífico', 'Europa', 'North America',
               'Greater China', 'Developing Markets', 'Global'],
    '2023': [61, 57, 58, 48, 49, 55],
    '2024': [77, 80, 82, 75, 72, 78]
})

fig_adopt = go.Figure()

fig_adopt.add_trace(go.Bar(
    x=df_adoption['Region'], y=df_adoption['2023'],
    name='2023', marker_color='rgba(44, 62, 80, 0.4)'
))
fig_adopt.add_trace(go.Bar(
    x=df_adoption['Region'], y=df_adoption['2024'],
    name='2024', marker_color='#2c3e50'
))

fig_adopt.update_layout(**BASE_LAYOUT, title=None, barmode='group')
fig_adopt.update_yaxes(title='% organizaciones que usan IA', ticksuffix='%')
add_title(fig_adopt, 'Adopción organizacional de IA por región',
          '2023 vs 2024 | % de organizaciones que reportan uso de IA')
add_source(fig_adopt,
           'McKinsey & Company Survey 2024 — Stanford HAI AI Index 2025')
fig_adopt.show()


**Figura 14 — EAP ausente del mapa global de demanda laboral de IA**

In [26]:
df_jobs = pd.DataFrame({
    'País': ['Singapore', 'Luxembourg', 'Hong Kong', 'UAE',
             'United States', 'Canada', 'Switzerland', 'Belgium',
             'Sweden', 'UK', 'Netherlands', 'Germany', 'Australia',
             'France', 'Austria', 'Italy', 'Mexico', 'Chile',
             'New Zealand', 'Croatia'],
    'ai_jobs_pct': [3.27, 1.99, 1.89, 1.72, 1.79, 1.41, 1.37,
                    1.31, 1.31, 1.26, 1.25, 1.15, 1.14, 1.10,
                    1.06, 0.87, 0.73, 0.65, 0.55, 0.13],
    'region': ['EAP', 'Europa', 'EAP', 'MENA', 'Norte América',
               'Norte América', 'Europa', 'Europa', 'Europa', 'Europa',
               'Europa', 'Europa', 'Oceanía', 'Europa', 'Europa',
               'Europa', 'LATAM', 'LATAM', 'Oceanía', 'Europa']
}).sort_values('ai_jobs_pct', ascending=True)

color_map = {'EAP': '#e74c3c', 'Norte América': '#2c3e50', 'Europa': '#95a5a6',
             'MENA': '#e67e22', 'LATAM': '#27ae60', 'Oceanía': '#3498db'}

fig_jobs = go.Figure()

fig_jobs.add_trace(go.Bar(
    x=df_jobs['ai_jobs_pct'], y=df_jobs['País'],
    orientation='h',
    marker_color=[color_map[r] for r in df_jobs['region']],
    showlegend=False
))

for region, color in color_map.items():
    fig_jobs.add_trace(go.Bar(
        x=[None], y=[None], name=region,
        marker_color=color, showlegend=True
    ))

fig_jobs.update_layout(**BASE_LAYOUT, title=None, height=600, barmode='overlay')
fig_jobs.update_xaxes(title='% ofertas de empleo que requieren habilidades de IA',
                      ticksuffix='%')
add_title(fig_jobs, 'Demanda laboral de IA: EAP ausente del mapa global',
          '% de ofertas de empleo que requieren habilidades de IA | 2024')
add_source(fig_jobs, 'Lightcast 2024 — Stanford HAI AI Index 2025')
fig_jobs.show()


Ningún país de la muestra EAP — Malaysia, Vietnam, Indonesia, Filipinas — 
aparece en el ranking global de demanda laboral de IA. Solo Singapur (3.27%) y 
Hong Kong (1.89%), ambas economías-ciudad atípicas, representan la región.

Esta ausencia no es un accidente metodológico — refleja que LinkedIn y las bolsas 
de empleo digitales tienen cobertura muy baja en estos mercados laborales. Si los 
países de EAP son invisibles en los datos de demanda laboral de IA, la brecha que 
documenta el Banco Mundial puede ser en parte un artefacto de la cobertura desigual 
de los sistemas de medición globales.


**Figura 15 — Impacto financiero de la IA por función de negocio**

In [27]:
df_impact = pd.DataFrame({
    'Función': ['Marketing y ventas', 'Supply chain', 'Service operations',
                'Software engineering', 'IT', 'Product development', 'HR', 'Risk y legal'],
    'cost_savings' : [25, 43, 49, 41, 37, 23, 37, 34],
    'revenue_gains': [71, 63, 57, 34, 28, 43, 20, 15]
}).sort_values('revenue_gains', ascending=True)

fig_impact = go.Figure()

fig_impact.add_trace(go.Bar(
    x=df_impact['cost_savings'], y=df_impact['Función'],
    orientation='h', name='Ahorro de costes',
    marker_color='rgba(44, 62, 80, 0.5)'
))
fig_impact.add_trace(go.Bar(
    x=df_impact['revenue_gains'], y=df_impact['Función'],
    orientation='h', name='Incremento de ingresos',
    marker_color='#2c3e50'
))

fig_impact.update_layout(**BASE_LAYOUT, title=None, barmode='overlay', height=500)
fig_impact.update_xaxes(title='% organizaciones que reportan impacto', ticksuffix='%')
add_title(fig_impact, 'Impacto financiero de la IA por función de negocio',
          '% organizaciones que reportan beneficios | Global 2024')
add_source(fig_impact, 'McKinsey & Company Survey 2024 — Stanford HAI AI Index 2025')
fig_impact.show()


Incluso donde hay adopción, los beneficios son predominantemente modestos — 
reducciones de coste inferiores al 10% e incrementos de ingresos inferiores al 5%. 
Para los países de EAP, acelerar la adopción de IA sin construir capacidad para 
extraer valor productivo real reproduce exactamente el patrón documentado en las 
exportaciones tecnológicas: alta intensidad superficial, bajo valor añadido doméstico.


## 6. Conclusiones

Este análisis, construido desde datos del WDI, patentes internacionales, 
productividad sectorial de OpenDOSM y el Stanford HAI AI Index, llega a tres 
conclusiones que cuestionan la narrativa del informe del Banco Mundial:

### 6.1 La apertura comercial, no la tecnología exportada, impulsa el crecimiento

El único regresor robusto a través de todas las especificaciones econométricas es 
`trade_openness` — consistentemente positivo y significativo al 5%. Las exportaciones 
tecnológicas, contra lo que sugiere el informe, tienen un efecto negativo y 
significativo sobre el crecimiento una vez controlada la heterogeneidad individual. 
Integración comercial sí; intensidad exportadora tecnológica, no necesariamente.

### 6.2 Exportar tecnología no equivale a ser tecnológicamente capaz

Malaysia exporta cinco veces más tecnología que Indonesia pero registra exactamente 
el mismo nivel de innovación doméstica medida por patentes. El sector eléctrico-
electrónico malayo — núcleo de sus exportaciones tecnológicas — lleva tres años sin 
ganar productividad. Los países de EAP son ensambladores sofisticados en cadenas 
de valor globales, no economías tecnológicamente capaces. El informe confunde el 
envoltorio con el contenido.

### 6.3 El Banco Mundial mide mal la adopción de IA

El 13–17% de adopción interna de IA que cita el informe corresponde a subsidiarias 
de multinacionales — las empresas más avanzadas tecnológicamente por definición. 
McKinsey y Stanford HAI reportan una adopción organizacional del 77% en Asia-
Pacífico para 2024. Las dos cifras no son comparables, y el informe no hace esa 
distinción. Adicionalmente, los mercados laborales de IA de los principales países 
de EAP son invisibles en los datos globales de Lightcast — lo que sugiere que la 
brecha documentada puede ser en parte un artefacto de cobertura, no una realidad 
económica verificada.

---

### Agenda de investigación

Tres extensiones naturales para trabajos futuros:

1. **Microdatos de empresa** — el Enterprise Survey del Banco Mundial permitiría 
   contrastar si el efecto negativo de `tech_exports` se concentra en empresas 
   ensambladores vs empresas innovadoras.
2. **Valor añadido doméstico en exportaciones** — los datos de UNCTAD sobre GVC 
   participation permitirían medir directamente el contenido doméstico de las 
   exportaciones tecnológicas.
3. **Panel ampliado** — incorporar Singapur, Corea del Sur y Taiwán ampliaría la 
   variación between y daría más potencia a los estimadores.

---

*Datos, código y metodología disponibles en [edalytics.com](https://ji-square.github.io/edalytics_website/)*  
*Fuentes: World Bank WDI, World Bank EAP Economic Update (Abril 2026), Stanford HAI AI Index 2025, OpenDOSM, Lightcast 2024*


In [28]:
models = {
    'FE Niveles'      : fe_result,
    'FD Parsimonioso' : fd_result,
    'FD Umbral'       : fd_result_threshold,
    'FD Política Ind.': fd_result_policy
}

for name, model in models.items():
    print(f"\n=== {name} ===")
    print(pd.concat([model.params, model.conf_int()], axis=1).round(3))


=== FE Niveles ===
                parameter  lower  upper
tech_exports       -0.131 -0.277  0.015
investment         -0.043 -0.246  0.159
inflation           0.256 -0.132  0.644
trade_openness      0.093  0.015  0.172

=== FD Parsimonioso ===
                parameter  lower  upper
tech_exports       -0.820 -1.482 -0.158
investment          0.801 -0.334  1.936
trade_openness      0.301  0.049  0.553

=== FD Umbral ===
                parameter  lower  upper
tech_exports       -1.028 -1.794 -0.263
investment          0.808 -0.309  1.925
trade_openness      0.294  0.030  0.558
tech_high           0.112  0.016  0.208

=== FD Política Ind. ===
                parameter  lower  upper
tech_exports       -1.124 -2.159 -0.088
investment          0.736 -0.270  1.741
trade_openness      0.294  0.058  0.529
tech_policy         0.772 -0.330  1.875


In [31]:
fig_comp.update_layout(**BASE_LAYOUT, title=None, height=500)
fig_comp.update_yaxes(
    tickmode='array',
    tickvals=list(range(len(var_order))),
    ticktext=[var_labels[v] for v in var_order],
    gridcolor='#e8e8e4',
    gridwidth=0.8,
    linecolor='#2c3e50',
    tickfont=dict(color='#2c3e50')
)
fig_comp.update_xaxes(title='Coeficiente estimado')
add_title(fig_comp,
          'Robustez de los coeficientes entre especificaciones',
          'Intervalos de confianza al 95% | ◆ significativo | ○ no significativo')
add_source(fig_comp, 'World Bank WDI | Estimación propia')
fig_comp.show()